# Plot multi-lead predictions for predicting hurricane track (distance) errors.
author: Elizabeth A. Barnes and Randal J. Barnes

In [1]:
%matplotlib inline
%load_ext autotime

import sys
import os
import importlib as imp
import warnings
from shapely.errors import ShapelyDeprecationWarning

warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning)

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cartopy as ct
import plots
import compute_predictions

import experiment_settings
import mahalanobis

mpl.rcParams['savefig.dpi'] = 600
mpl.rcParams["figure.dpi"] = 100
dpiFig = 600
plots.set_plot_rc()
warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)

time: 4.02 s (started: 2023-12-07 16:04:37 -07:00)


In [2]:
__author__ = "Randal J Barnes and Elizabeth A. Barnes"
__version__ = "16 December 2022"

time: 141 µs (started: 2023-12-07 16:04:41 -07:00)


In [3]:
testing = experiment_settings.Experiments()
print('index range and short names for experiments')
for expname in testing.get_exp_list_short:
    exp_inds = [index for index, exp_string in enumerate(testing.get_exp_list) if expname in exp_string]
    print(np.min(exp_inds), '-', np.max(exp_inds)+1, expname)

index range and short names for experiments
0 - 20 dev_OFD
20 - 40 dev_OBD
time: 1.47 ms (started: 2023-12-07 16:04:41 -07:00)


In [4]:
# pick the experiment
expname = 'dev_OFD'

# get the correct indices for these experiments
exp_inds = [index for index, exp_string in enumerate(testing.get_exp_list) if expname in exp_string]

time: 203 µs (started: 2023-12-07 16:04:41 -07:00)


In [5]:
EXP_NAME_VEC = testing.get_exp_list[np.min(exp_inds):np.max(exp_inds)+1]
EXP_NAME_VEC

['dev_OFD_AL12',
 'dev_OFD_AL24',
 'dev_OFD_AL36',
 'dev_OFD_AL48',
 'dev_OFD_AL60',
 'dev_OFD_AL72',
 'dev_OFD_AL84',
 'dev_OFD_AL96',
 'dev_OFD_AL108',
 'dev_OFD_AL120',
 'dev_OFD_EP12',
 'dev_OFD_EP24',
 'dev_OFD_EP36',
 'dev_OFD_EP48',
 'dev_OFD_EP60',
 'dev_OFD_EP72',
 'dev_OFD_EP84',
 'dev_OFD_EP96',
 'dev_OFD_EP108',
 'dev_OFD_EP120']

time: 2.85 ms (started: 2023-12-07 16:04:41 -07:00)


In [6]:
DATA_PATH = "data/"
MODEL_PATH = os.path.join("saved_models", expname)
FIGURE_PATH = os.path.join("figures/analysis/", expname)
PREDICTIONS_PATH = os.path.join("saved_predictions/", expname)

if not os.path.exists(FIGURE_PATH):
    os.makedirs(FIGURE_PATH)

time: 338 µs (started: 2023-12-07 16:04:41 -07:00)


# Plot Results

In [7]:
storm_dict = {
    "IAN": {"storm_name": "IAN", "year": 2022, "basin": "AL"},
    # https://www.nhc.noaa.gov/aboutcone.shtml
    "FIONA": {"storm_name": "FIONA", "year": 2022, "basin": "AL"},
    "IRMA": {"storm_name": "IRMA", "year": 2017, "basin": "AL"},
    # https://www.air-worldwide.com/blog/posts/2017/8/the-ever-shrinking-cone-of-uncertainty/
    "NICOLE": {"storm_name": "NICOLE", "year": 2022, "basin": "AL"},
    "JULIA": {"storm_name": "JULIA", "year": 2022, "basin": "AL"},
    "NORMAN": {"storm_name": "NORMAN", "year": 2018, "basin": "EP"},
    "HARVEY": {"storm_name": "HARVEY", "year": 2017, "basin": "AL"},
    "DORIAN": {"storm_name": "DORIAN", "year": 2019, "basin": "AL"},
    "OTIS": {"storm_name": "OTIS", "year": 2023, "basin": "EP"},
    "PHILIPPE": {"storm_name": "PHILIPPE", "year": 2023, "basin": "AL"},
    }

# averaged for 60, 84, and 108 hours (up to 2021 for 60, all years for 84, 108)
cone_of_uncertainty = {
    "2013": {"AL": {0:8, 12:33, 24:52, 36:72, 48:92, 60:110, 72:128, 84:152, 96:177, 108:203, 120:229}, "EP": {0:8, 12:30, 24:49, 36:66, 48:82, 60:96, 72:111, 84:134, 96:157, 108:177, 120:197}},
    "2014": {"AL": {0:8, 12:33, 24:52, 36:72, 48:92, 60:108, 72:125, 84:147, 96:170, 108:198, 120:226}, "EP": {0:8, 12:30, 24:46, 36:62, 48:79, 60:92, 72:105, 84:129, 96:154, 108:172, 120:190}},
    "2015": {"AL": {0:8, 12:32, 24:52, 36:71, 48:90, 60:106, 72:122, 84:146, 96:170, 108:197, 120:225}, "EP": {0:8, 12:26, 24:42, 36:54, 48:69, 60:84, 72:100, 84:121, 96:143, 108:162, 120:182}},
    "2016": {"AL": {0:8, 12:30, 24:49, 36:66, 48:84, 60:99, 72:115, 84:140, 96:165, 108:201, 120:237}, "EP": {0:8, 12:27, 24:42, 36:55, 48:70, 60:85, 72:100, 84:118, 96:137, 108:154, 120:172}},
    "2017": {"AL": {0:8, 12:29, 24:45, 36:63, 48:78, 60:92, 72:107, 84:133, 96:159, 108:185, 120:211}, "EP": {0:8, 12:25, 24:40, 36:51, 48:66, 60:79, 72:93, 84:104, 96:116, 108:133, 120:151}},
    "2018": {"AL": {0:8, 12:26, 24:43, 36:56, 48:74, 60:88, 72:103, 84:127, 96:151, 108:174, 120:198}, "EP": {0:8, 12:25, 24:39, 36:50, 48:66, 60:80, 72:94, 84:109, 96:125, 108:143, 120:162}},
    "2019": {"AL": {0:8, 12:26, 24:41, 36:54, 48:68, 60:85, 72:102, 84:126, 96:151, 108:174, 120:198}, "EP": {0:8, 12:25, 24:38, 36:48, 48:62, 60:75, 72:88, 84:101, 96:115, 108:130, 120:145}},
    "2020": {"AL": {0:8, 12:26, 24:41, 36:55, 48:69, 60:86, 72:103, 84:127, 96:151, 108:173, 120:196}, "EP": {0:8, 12:25, 24:38, 36:51, 48:65, 60:78, 72:91, 84:103, 96:115, 108:126, 120:138}},
    "2021": {"AL": {0:8, 12:27, 24:40, 36:55, 48:69, 60:86, 72:102, 84:125, 96:148, 108:174, 120:200}, "EP": {0:8, 12:25, 24:37, 36:51, 48:64, 60:77, 72:89, 84:101, 96:114, 108:126, 120:138}},
    "2022": {"AL": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 84:121, 96:142, 108:171, 120:200}, "EP": {0:8, 12:25, 24:38, 36:51, 48:65, 60:79, 72:93, 84:106, 96:120, 108:133, 120:146}},
    "2023": {"AL": {0:8, 12:26, 24:39, 36:53, 48:67, 60:81, 72:99, 84:122, 96:145, 108:175, 120:205}, "EP": {0:8, 12:25, 24:38, 36:51, 48:63, 60:78, 72:86, 84:98, 96:110, 108:123, 120:137}}
    }

time: 1.45 ms (started: 2023-12-07 16:04:41 -07:00)


In [ ]:
imp.reload(mahalanobis)
imp.reload(plots)
imp.reload(compute_predictions)
import glob
KM_TO_DEG = 1.0 / 111.

for storm_name in ("PHILIPPE",):#("IAN","IRMA","FIONA"):#("IAN", "NICOLE", "IRMA"):
    print(storm_name)
    storm = storm_dict[storm_name]

    for RNG_SEED in testing.get_experiment(EXP_NAME_VEC[0])['rng_seed_list']:

        TESTING_YEAR = storm["year"]
        files = glob.glob(PREDICTIONS_PATH + '/*')
        
        # GET PREDICTIONS
        df_pred_test = pd.DataFrame()
        for file in files:
            try:
                df = pd.read_csv(file)
            except:
                continue

            rng_seed = RNG_SEED
            years_test = (TESTING_YEAR,)
            df["exp_name"] = expname
            df_pred_test = pd.concat([df_pred_test, df], axis=0)

        #------------------------------------------------------------
        # MAKE THE PLOTS
        df = df_pred_test.loc[
            (df_pred_test["NAME"] == storm_name) & (df_pred_test["YEAR"] == TESTING_YEAR)].copy()
        df = df.sort_values("MMDDHH").reset_index(drop=True)
        forecast_dates = df["MMDDHH"].unique()

        for i, pred_time in enumerate(forecast_dates):
            print(str(i+1) + ' of ' + str(len(forecast_dates)) + ': ' + str(pred_time))
            storm["pred_time"] = pred_time
            df_storm = df_pred_test.loc[
                (df_pred_test["NAME"] == storm_name) & (df_pred_test["MMDDHH"] == storm["pred_time"])].copy()
            df_storm = compute_predictions.add_lead_zero(df_storm)
            df_storm["nhc_cone_radius"] = [cone_of_uncertainty[str(storm['year'])][storm['basin']][key] for key in df_storm["FHOUR"].unique()]

            # get dynamic extent
            extx = [df_storm["LONN"], df_storm["LONN"] + KM_TO_DEG * df_storm["OFDX"]]
            exty = [df_storm["LATN"], df_storm["LATN"] + KM_TO_DEG * df_storm["OFDY"]]
            storm_extent = [-(360-np.min(extx)+10), -(360-np.max(extx)-10), np.min(exty)-10, np.max(exty)+10]
            
            # plot probability ellipses
            fig = plt.figure(dpi=150, )
            ax = fig.add_subplot(1, 1, 1, projection=ct.crs.PlateCarree(central_longitude=0.))
            details = plots.plot_probability_ellipses(
                df_storm,
                ax=ax,
                leadtimes=np.arange(0,120+12, 12),
                contours=(.1, .25, .5, .75, .9,),
                extent = storm_extent,
                alpha=.4,
                vector=True,
                plot_nhc_cone=False,
            )
            plt.gca().get_legend().remove()
            ax.set_extent(storm_extent, crs=ct.crs.PlateCarree())
            plt.savefig(
                FIGURE_PATH + '/probability_ellipses_rng_seed_' + str(RNG_SEED) + '_' + details.replace(' ', '_') + '.png',
                dpi=dpiFig,
                bbox_inches='tight',
            )
            plt.close()

            # plot banana cones
            try:
                fig = plt.figure(dpi=150, )
                ax = fig.add_subplot(1, 1, 1, projection=ct.crs.PlateCarree(central_longitude=0.))
                details = plots.plot_banana_of_uncertainty(
                    df_storm=df_storm,
                    ax=ax,
                    extent=storm_extent,
                    vector=True,
                    colors=("steelblue","khaki"),
                    alpha=.75,
                    plot_nhc_cone=True,
                )
                ax.set_extent(storm_extent, crs=ct.crs.PlateCarree())
                plt.savefig(
                    FIGURE_PATH + '/banana_cone_rng_seed_' + str(RNG_SEED) + '_' + details.replace(' ', '_') + '.png',
                    dpi=dpiFig,
                    bbox_inches='tight',
                )
                plt.close()
            except:
                print('not enough data for spline computation. not making the figure.')
                plt.close()

In [33]:
imp.reload(mahalanobis)
imp.reload(plots)
imp.reload(compute_predictions)
import glob
KM_TO_DEG = 1.0 / 111.

for storm_name in ('BRET', 'FRANKLIN', 'IDALIA', 'PHILIPPE',):
    print(storm_name)

    for RNG_SEED in testing.get_experiment(EXP_NAME_VEC[0])['rng_seed_list']:

        TESTING_YEAR = 2023
        files = glob.glob(PREDICTIONS_PATH + '/*')
        
        # GET PREDICTIONS
        df_pred_test = pd.DataFrame()
        for file in files:
            try:
                df = pd.read_csv(file)
            except:
                continue

            rng_seed = RNG_SEED
            years_test = (TESTING_YEAR,)
            df["exp_name"] = expname
            df_pred_test = pd.concat([df_pred_test, df], axis=0)

        df_pred_test = df_pred_test[df_pred_test['FHOUR'] == 48]

        #------------------------------------------------------------
        # MAKE THE PLOTS
        df = df_pred_test.loc[
            (df_pred_test["NAME"] == storm_name) & (df_pred_test["YEAR"] == TESTING_YEAR)].copy()
        df = df.sort_values("MMDDHH").reset_index(drop=True)
        forecast_dates = df["MMDDHH"].unique()

        for i, pred_time in enumerate(forecast_dates):
            print(str(i+1) + ' of ' + str(len(forecast_dates)) + ': ' + str(pred_time))
            df_storm = df_pred_test.loc[
                (df_pred_test["NAME"] == storm_name) & (df_pred_test["MMDDHH"] == pred_time)].copy()
            df_storm["nhc_cone_radius"] = [cone_of_uncertainty[str(2023)]['AL'][key] for key in df_storm["FHOUR"].unique()]

            # get dynamic extent
            extx = [df_storm["LONN"], df_storm["LONN"] + KM_TO_DEG * df_storm["OFDX"]]
            exty = [df_storm["LATN"], df_storm["LATN"] + KM_TO_DEG * df_storm["OFDY"]]
            storm_extent = [-(360-np.min(extx[1:])+5), -(360-np.max(extx[1:])-5), np.min(exty[1:])-5, np.max(exty[1:])+5]
            
            # plot probability ellipses
            fig = plt.figure(dpi=150, )
            ax = fig.add_subplot(1, 1, 1, projection=ct.crs.PlateCarree(central_longitude=0.))
            details = plots.plot_probability_ellipses(
                df_storm,
                ax=ax,
                leadtimes=[48,],
                contours=(2/3,),
                annotate_leadtimes=False,
                extent = storm_extent,
                alpha=0.4,
                vector=True,
                plot_nhc_cone=True,
            )
            plt.gca().get_legend().remove()
            ax.set_extent(storm_extent, crs=ct.crs.PlateCarree())
            plt.savefig(
                FIGURE_PATH + '/simple_ellipses_' + details.replace(' ', '_') + '.png',
                dpi=dpiFig,
                bbox_inches='tight',
            )
            plt.close()

BRET
1 of 13: 61918
2 of 13: 62000
3 of 13: 62006
4 of 13: 62012
5 of 13: 62018
6 of 13: 62100
7 of 13: 62106
8 of 13: 62112
9 of 13: 62118
10 of 13: 62200
11 of 13: 62206
12 of 13: 62212
13 of 13: 62218
CINDY
1 of 7: 62212
2 of 7: 62218
3 of 7: 62300
4 of 7: 62306
5 of 7: 62312
6 of 7: 62318
7 of 7: 62400
DON
1 of 29: 71412
2 of 29: 71418
3 of 29: 71500
4 of 29: 71512
5 of 29: 71518
6 of 29: 71600
7 of 29: 71606
8 of 29: 71618
9 of 29: 71700
10 of 29: 71706
11 of 29: 71712
12 of 29: 71718
13 of 29: 71800
14 of 29: 71806
15 of 29: 71812
16 of 29: 71900
17 of 29: 71906
18 of 29: 71912
19 of 29: 71918
20 of 29: 72000
21 of 29: 72006
22 of 29: 72012
23 of 29: 72018
24 of 29: 72100
25 of 29: 72106
26 of 29: 72112
27 of 29: 72118
28 of 29: 72200
29 of 29: 72206
FRANKLIN
1 of 38: 82100
2 of 38: 82106
3 of 38: 82112
4 of 38: 82200
5 of 38: 82206
6 of 38: 82212
7 of 38: 82218
8 of 38: 82300
9 of 38: 82306
10 of 38: 82312
11 of 38: 82318
12 of 38: 82400
13 of 38: 82406
14 of 38: 82412
15 of 38: